# Dự báo nhu cầu thuê xe theo giờ: ML hay một bảng trung bình?

**Bối cảnh.** Operations Manager của hệ thống xe đạp công cộng lên lịch xe tải điều phối xe và nhân viên trực theo
**dự báo số lượt thuê từng giờ của ngày mai**. Hiện tại đội dùng một bảng trung bình theo (ngày làm việc hay không,
giờ trong ngày). Một vendor chào một model ML "giảm sai số một nửa". Ops Manager hỏi bạn: *"Có đúng không? Model đó cần
dữ liệu gì mà chúng ta chưa có? Có đáng mua không?"*

Dữ liệu: bộ `bike-sharing-uci`, `hour.csv` (Capital Bikeshare, Washington D.C., 2011–2012, 17 379 giờ), giấy phép CC BY
4.0. Cột: `dteday`, `hr`, `workingday`, `weekday`, `mnth`, `yr`, `holiday`, `season`, thời tiết **đo thực tế** (`weathersit`,
`temp`, `atemp`, `hum`, `windspeed`, đã chuẩn hoá), và `cnt` (lượt thuê). Một số giờ bị thiếu hẳn trong tệp (ví dụ
29–30/10/2012 chỉ còn vài giờ): kiểm độ phủ trước khi đọc sai số của những ngày đó.

Kiểm tra theo cách dự báo được dùng: học trên dữ liệu **trước** 01/10/2012, đo trên 01/10–31/12/2012.

**Phần được chấm tự động**: ba hàm bên dưới và khung `MEMO.md`. Phần còn lại là phân tích tự do, chấm theo rubric.

Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/advanced/02-bike-ml-or-baseline -q`.

In [ ]:
# explore: dữ liệu và độ phủ giờ
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

hour = pd.read_csv(ensure("bike-sharing-uci") / "hour.csv", parse_dates=["dteday"])
per_day = hour.groupby("dteday").size()
print(hour.shape, hour.dteday.min().date(), "→", hour.dteday.max().date())
print("ngày thiếu giờ:", (per_day < 24).sum(), "; ngày thiếu nhiều nhất trong kỳ test:")
print(per_day[per_day < 20].loc["2012-10-01":])

In [ ]:
import numpy as np
import pandas as pd

## Ba hàm được chấm

`time_split(hour, cutoff) -> (train, test)`: `train` là các dòng `dteday < cutoff`, `test` là `dteday >= cutoff` (giữ index
gốc, không sửa đầu vào). Phần nào rỗng thì `ValueError`.

`profile_baseline(train, test, weeks=None) -> Series`: dự báo cho từng dòng của `test` (cùng index, tên `pred`) bằng mean
`cnt` của `train` theo cặp (`workingday`, `hr`). Khi `weeks` là số, chỉ dùng các dòng `train` có
`dteday > train.dteday.max() - weeks tuần` (tức `weeks` tuần cuối). Cặp không có trong phần train được dùng thì dự báo bằng
mean `cnt` của phần đó.

`mae(y_true, y_pred) -> float`: sai số tuyệt đối trung bình; `ValueError` khi khác độ dài, rỗng hoặc có NaN.

In [ ]:
def time_split(hour: pd.DataFrame, cutoff) -> tuple[pd.DataFrame, pd.DataFrame]:
    """train: dteday < cutoff; test: dteday >= cutoff."""
    # TODO
    raise NotImplementedError


def profile_baseline(train: pd.DataFrame, test: pd.DataFrame, weeks: int | None = None) -> pd.Series:
    """Mean cnt theo (workingday, hr) của train (hoặc weeks tuần cuối), gán cho từng dòng test."""
    # TODO: chọn phần train; groupby(["workingday", "hr"]).cnt.mean(); join vào test; NaN → mean chung
    raise NotImplementedError


def mae(y_true, y_pred) -> float:
    """Mean absolute error; ValueError khi khác độ dài, rỗng hoặc có NaN."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: baseline đang dùng, hai biến thể
train, test = time_split(hour, "2012-10-01")
print(f"train {len(train):,} giờ, test {len(test):,} giờ, lượt thuê TB/giờ trong test {test.cnt.mean():.1f}")
for weeks in [None, 8]:
    print(f"baseline (weeks={weeks}): MAE {mae(test.cnt, profile_baseline(train, test, weeks)):.2f}")

In [ ]:
# explore: model kiểu vendor (gradient boosting), hai bộ đặc trưng
from sklearn.ensemble import HistGradientBoostingRegressor

CALENDAR = ["hr", "workingday", "weekday", "mnth", "yr", "holiday", "season"]
WEATHER = ["weathersit", "temp", "atemp", "hum", "windspeed"]
for name, cols in [("lịch", CALENDAR), ("lịch + thời tiết đo thực tế", CALENDAR + WEATHER)]:
    model = HistGradientBoostingRegressor(random_state=0).fit(train[cols], train.cnt)
    print(f"ML, {name}: MAE {mae(test.cnt, model.predict(test[cols])):.2f}")

## Phân tích tự do (gợi ý các bước)

1. **Dữ liệu có sẵn lúc dự báo**: model dùng thời tiết **đo thực tế** của chính giờ đó. Lúc lên lịch tối hôm trước, bạn chỉ
   có **dự báo** thời tiết. Ước lượng model mất bao nhiêu khi thời tiết kém chính xác hơn (ví dụ: dùng thời tiết của cùng
   giờ ngày hôm trước làm "dự báo ngây thơ").
2. **Sai số ở đâu**: MAE theo tháng, theo giờ cao điểm / thấp điểm, theo `weathersit`. Ngày nào sai nhiều nhất, có phải
   ngày thiếu giờ không?
3. **Baseline có thể tốt hơn không** trước khi mua ML: thêm thời tiết vào bảng trung bình (ví dụ hệ số cho ngày mưa), dùng
   cùng giờ tuần trước.
4. **Quy ra vận hành**: sai số 1 lượt/giờ có nghĩa gì với xe tải điều phối? Giờ nào sai số đắt nhất?

Rồi viết `starter/MEMO.md` theo khung có sẵn.

In [ ]:
# explore: chỗ làm việc của bạn